# Spark + Polaris Iceberg Demo

This notebook demonstrates **multi-engine access** to Iceberg tables through Apache Polaris.
The same tables created by Trino are readable (and writable) from Spark — both engines share one catalog.

## 1. Create SparkSession
The Polaris REST catalog and S3 settings are pre-configured in `spark-defaults.conf`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Polaris Iceberg Demo") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Default catalog: {spark.conf.get('spark.sql.defaultCatalog')}")

Spark version: 3.5.0
Default catalog: polaris


## 2. Browse the Polaris Catalog
List namespaces and tables — these were created by Trino.

In [2]:
spark.sql("SHOW NAMESPACES IN polaris").show()

+---------+
|namespace|
+---------+
|     shop|
+---------+



In [3]:
spark.sql("SHOW TABLES IN polaris.shop").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|     shop|     orders|      false|
|     shop|order_items|      false|
+---------+-----------+-----------+



## 3. Query Iceberg Tables (created by Trino)
Spark reads the same Parquet files on S3 via the shared Polaris catalog.

In [4]:
orders = spark.sql("SELECT * FROM polaris.shop.orders ORDER BY order_date")
orders.show(20, truncate=False)

+--------+-----------+----------+---------+------------+
|order_id|customer_id|order_date|status   |total_amount|
+--------+-----------+----------+---------+------------+
|1001    |1          |2024-01-15|completed|299.99      |
|1002    |2          |2024-01-16|completed|149.50      |
|1003    |3          |2024-01-17|shipped  |549.00      |
|1004    |1          |2024-01-18|completed|89.99       |
|1005    |5          |2024-01-19|pending  |1299.00     |
|1006    |7          |2024-01-20|completed|199.99      |
|1007    |4          |2024-01-21|shipped  |449.50      |
|1008    |10         |2024-01-22|completed|79.99       |
|1009    |6          |2024-01-23|cancelled|329.00      |
|1010    |8          |2024-01-24|completed|599.99      |
|1011    |12         |2024-02-01|completed|159.99      |
|1012    |15         |2024-02-03|shipped  |899.00      |
|1013    |9          |2024-02-05|completed|249.50      |
|1014    |11         |2024-02-07|pending  |179.99      |
|1015    |20         |2024-02-1

In [5]:
items = spark.sql("SELECT * FROM polaris.shop.order_items")
items.show(30, truncate=False)

+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|1      |1001    |5         |1       |299.99    |
|2      |1002    |3         |2       |74.75     |
|3      |1003    |8         |1       |549.00    |
|4      |1004    |2         |1       |89.99     |
|5      |1005    |10        |1       |1299.00   |
|6      |1006    |6         |1       |199.99    |
|7      |1007    |9         |1       |449.50    |
|8      |1008    |1         |1       |79.99     |
|9      |1009    |7         |1       |329.00    |
|10     |1010    |10        |1       |599.99    |
|11     |1001    |3         |1       |0.00      |
|12     |1002    |1         |1       |0.00      |
|13     |1003    |6         |2       |0.00      |
|14     |1006    |2         |3       |0.00      |
|15     |1007    |4         |1       |0.00      |
|16     |1010    |5         |1       |0.00      |
|17     |1011    |11        |1       |159.99    |


## 4. DataFrame API — Analytics
Use Spark's DataFrame API for aggregations.

In [6]:
from pyspark.sql import functions as F

# Orders by status
orders.groupBy("status") \
    .agg(
        F.count("*").alias("count"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .show()

+---------+-----+-------------+---------------+
|   status|count|total_revenue|avg_order_value|
+---------+-----+-------------+---------------+
|completed|    9|      2527.94|     280.882222|
|  shipped|    3|      1897.50|     632.500000|
|  pending|    2|      1478.99|     739.495000|
|cancelled|    1|       329.00|     329.000000|
+---------+-----+-------------+---------------+



## 6. Iceberg Table Metadata
Inspect Iceberg-specific metadata using the DataFrame API.

In [ ]:
# Snapshot history
spark.read.format("iceberg").load("polaris.shop.orders.snapshots").show(truncate=False)

In [ ]:
# Data files backing the orders table
spark.read.format("iceberg").load("polaris.shop.orders.files").select("file_path", "file_format", "record_count", "file_size_in_bytes").show(truncate=False)

## 6. Iceberg Table Metadata
Spark can inspect Iceberg-specific metadata: snapshots, history, and files.

In [6]:
# Snapshot history
spark.sql("SELECT * FROM polaris.shop.orders.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                            |summary                                     

In [9]:
# Snapshot history
spark.sql("SELECT * FROM polaris.shop.orders.snapshots").show(truncate=False)

Py4JJavaError: An error occurred while calling o44.sql.
: org.apache.iceberg.exceptions.NotAuthorizedException: Not authorized: 
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:210)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:118)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:102)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:211)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:323)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:262)
	at org.apache.iceberg.rest.HTTPClient.get(HTTPClient.java:358)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:96)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadInternal(RESTSessionCatalog.java:438)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadTable(RESTSessionCatalog.java:462)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.loadTable(BaseSessionCata

In [10]:
spark.sql("SELECT * FROM  polaris.earthquakes.events").show(truncate=False)

+-------------+---------+-----------------------------------------+-----------------------+-----------------------+-----------------+----------------+----------------+-------+----------+-------------------------------------------------+---------+--------------+------------+---------------------------------------------------------------+------------------+--------------+------------+
|earthquake_id|magnitude|place                                    |event_time             |updated_time           |longitude        |latitude        |depth_km        |tsunami|event_type|title                                            |status   |magnitude_type|significance|url                                                            |_dlt_load_id      |_dlt_id       |felt_reports|
+-------------+---------+-----------------------------------------+-----------------------+-----------------------+-----------------+----------------+----------------+-------+----------+------------------------------------------

In [10]:
# Data files backing the orders table
spark.sql("SELECT file_path, file_format, record_count, file_size_in_bytes FROM polaris.shop.orders.files").show(truncate=False)

Py4JJavaError: An error occurred while calling o44.sql.
: org.apache.iceberg.exceptions.NotAuthorizedException: Not authorized: 
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:210)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:118)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:102)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:211)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:323)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:262)
	at org.apache.iceberg.rest.HTTPClient.get(HTTPClient.java:358)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:96)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadInternal(RESTSessionCatalog.java:438)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadTable(RESTSessionCatalog.java:462)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.loadTable(BaseSessionCata